In [1]:
import xarray as xr
import dask
from dask_jobqueue import PBSCluster
from dask.distributed import Client

In [2]:
# chunks='auto' appproaches this value
dask.config.set({'array.chunk-size': '8 GiB'})
# set cluster
cluster = PBSCluster(
    cores=1, # The number of cores you want
    memory='32GB', # Amount of memory (resource_spec is the one)
    processes=1, # How many processes
    queue='casper', # The type of queue to utilize (/glade/u/apps/dav/opt/usr/bin/execcasper)
    local_directory='$TMPDIR', # Use your local directory
    account='P93300313', # Input your project ID here
    walltime='02:00:00', # Amount of wall time
)
cluster.scale(jobs=5)
client = Client(cluster)
client

Connection method: Cluster object,Cluster type: dask_jobqueue.PBSCluster
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/acruz/Analysis/proxy/8787/status,
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/acruz/Analysis/proxy/8787/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://128.117.208.182:38103,Workers: 0
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/acruz/Analysis/proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B


In [3]:
ds = xr.open_dataset('/glade/work/acruz/CESM21PRECT.nc', chunks='auto')
ds

<xarray.Dataset> Size: 28GB
Dimensions:  (member: 100, time: 1260, lat: 192, lon: 288)
Coordinates:
  * time     (time) object 10kB 1910-02-01 00:00:00 ... 2015-01-01 00:00:00
  * lat      (lat) float64 2kB -90.0 -89.06 -88.12 -87.17 ... 88.12 89.06 90.0
  * lon      (lon) float64 2kB 0.0 1.25 2.5 3.75 5.0 ... 355.0 356.2 357.5 358.8
Dimensions without coordinates: member
Data variables:
    PRECT    (member, time, lat, lon) float32 28GB dask.array<chunksize=(75, 900, 140, 210), meta=np.ndarray>

In [4]:
clim = ds.groupby('time.month').mean(dim='time')
clim = clim.chunk(chunks='auto')
clim

<xarray.Dataset> Size: 265MB
Dimensions:  (month: 12, member: 100, lat: 192, lon: 288)
Coordinates:
  * month    (month) int64 96B 1 2 3 4 5 6 7 8 9 10 11 12
  * lat      (lat) float64 2kB -90.0 -89.06 -88.12 -87.17 ... 88.12 89.06 90.0
  * lon      (lon) float64 2kB 0.0 1.25 2.5 3.75 5.0 ... 355.0 356.2 357.5 358.8
Dimensions without coordinates: member
Data variables:
    PRECT    (month, member, lat, lon) float32 265MB dask.array<chunksize=(12, 100, 192, 288), meta=np.ndarray>

In [5]:
anom = ds.groupby('time.month') - clim
anom = anom.chunk(chunks='auto')
anom

<xarray.Dataset> Size: 28GB
Dimensions:  (lat: 192, lon: 288, time: 1260, member: 100)
Coordinates:
  * lat      (lat) float64 2kB -90.0 -89.06 -88.12 -87.17 ... 88.12 89.06 90.0
  * lon      (lon) float64 2kB 0.0 1.25 2.5 3.75 5.0 ... 355.0 356.2 357.5 358.8
  * time     (time) object 10kB 1910-02-01 00:00:00 ... 2015-01-01 00:00:00
    month    (time) int64 10kB dask.array<chunksize=(1260,), meta=np.ndarray>
Dimensions without coordinates: member
Data variables:
    PRECT    (member, time, lat, lon) float32 28GB dask.array<chunksize=(100, 384, 192, 288), meta=np.ndarray>

In [6]:
anom.to_zarr('/glade/work/acruz/CESM/anomalies/PRECTA', mode='w', consolidated=True)

/glade/u/home/acruz/.conda/envs/analysis_11_20_2025/lib/python3.11/site-packages/zarr/api/asynchronous.py:247: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


In [7]:
client.shutdown()